In [1]:
import os
from datetime import datetime
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
# from gymnasium.wrappers.frame_stack import FrameStack/
from gymnasium.wrappers import FrameStackObservation
from environment import get_env
from model import VDS_Nav_CNN, get_volumetric_observation_space

# paths
filename = datetime.now().strftime("vds_nav_%Y%m%d_%H%M%S")
save_path = os.path.join("results", filename)
os.makedirs(save_path, exist_ok=True)

/home/avishkar/gym-pybullet-drones/gym_pybullet_drones/envs/BaseAviary.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
pybullet build time: Jan 29 2025 23:16:28


In [2]:
# config

height, width = 96, 96
n = 3 # seq length of depth images
max_depth = 1 # coz hopefully, all depth distances are normalize
train_env = get_env((height, width))

print("#############################")

print("Declared observation space:", train_env.observation_space.shape)

# train_env.observation_space = get_volumetric_observation_space(height, width, n, max_depth)
train_env = FrameStackObservation(train_env, 3)

print("Sanity Check")

obs, _ = train_env.reset()

print("Train env observation space:", train_env.observation_space.shape)
print("Obs shape:", obs.shape)

print("#############################")

policy_kwargs = {
    "features_extractor_class" : VDS_Nav_CNN,
    "features_extractor_kwargs" : {
            "out_dim":4
        }

}
model = PPO(
    "CnnPolicy", # Although we use custom extractor, we start from CnnPolicy or MultiInputPolicy
    train_env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    tensorboard_log=os.path.join(save_path, "tb_logs"),
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01
)

checkpoint_callback = CheckpointCallback(
        save_freq=10000, 
        save_path=os.path.join(save_path, "checkpoints"),
        name_prefix="vds_model"
    )

[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
viewMatrix (-0.8660253882408142, -0.2499999701976776, 0.4330126941204071, 0.0, 0.0, 0.8660253286361694, 0.4999999701976776, 0.0, -0.4999999701976776, 0.4330126643180847, -0.75, 0.0, -0.0, 5.960464477539063e-08, -2.999999761581421, 1.0)
projectionMatrix (1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, -1.0000200271606445, -1.0, 0.0, 0.0, -0.02000020071864128, 0.0)


/home/avishkar/.local/lib/python3.10/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/avishkar/.local/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


#############################
Declared observation space: (96, 96, 1)
Sanity Check
Train env observation space: (3, 96, 96, 1)
Obs shape: (3, 96, 96, 1)
#############################
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [3]:
print("Starting training...")
try:
    model.learn(
        total_timesteps=1000000,
        callback=checkpoint_callback,
        tb_log_name="PPO_VDS"
    )
except KeyboardInterrupt:
    print("Training interrupted.")
finally:
    # Save final model
    model.save(os.path.join(save_path, "vds_final_model"))
    print(f"Model saved to {save_path}")

Starting training...
Logging to results/vds_nav_20260321_102155/tb_logs/PPO_VDS_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 250      |
|    ep_rew_mean     | -7.8e+03 |
| time/              |          |
|    fps             | 27       |
|    iterations      | 1        |
|    time_elapsed    | 75       |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 250          |
|    ep_rew_mean          | -7.81e+03    |
| time/                   |              |
|    fps                  | 25           |
|    iterations           | 2            |
|    time_elapsed         | 159          |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0044619674 |
|    clip_fraction        | 0.0236       |
|    clip_range           | 0.2          |
|    entropy_loss  